# 05. Precálculo de ELMo

ELMo (`simple_elmo`) corre en TensorFlow y el encoder del notebook 06_deep_encoder en PyTorch. Para que los
dos frameworks no compartan la GPU durante el entrenamiento, ELMo se calcula aquí una sola vez y
el notebook 06_deep_encoder solo lee los vectores guardados (sin importar TensorFlow).

- **Salida**: un vector por token, la misma que se usaba al vuelo: media de las 3 capas de ELMo,
  1024 dimensiones (como en el notebook 03). Se guarda en float16; al leer se convierte a float32.
- **Entrada**: ELMo es contextual, así que se calcula sobre la secuencia ya truncada, exactamente
  la que ve el modelo. Por eso hay una caché por configuración de entrada:
  - `head_tail__L1024` (configuración base, primeros 512 + últimos 512): siempre.
  - `head__L1024` y `head_tail__L512` (ablaciones): solo si ELMo gana el paso 1 del notebook 06_deep_encoder,
    que lee este notebook de los resultados guardados, o con `FORCE_ABLATIONS = True`.
- **Estado del LSTM**: el biLM de `simple_elmo` es *stateful* (el estado final de cada fila pasa
  como estado inicial al batch siguiente). Aquí se reinicia a cero antes de cada batch y, como cada
  fila tiene su propio estado, el vector de un prompt no depende de lo procesado antes ni de los
  otros prompts del batch (se comprueba al final).
- **Formato** (`outputs/elmo_cache/<configuración>/<split>/`, ignorado por git): `vectors.f16`
  es la matriz (tokens, 1024) en float16 sin cabecera, `index.npz` guarda para cada prompt su id,
  su fila de inicio y su nº de tokens, y `progress.json` el avance. Se lee con `np.memmap`, así que
  los vectores de un prompt se leen por su id sin cargar todo en memoria.
- **Reanudable**: tras cada batch se vuelcan los datos a disco y se guarda el progreso; si se
  interrumpe, al relanzar el notebook continúa desde el último batch completo. Los prompts se
  procesan siempre en el mismo orden: de más largo a más corto, para que un posible fallo de
  memoria aparezca al principio.

Hay que ejecutarlo con el mismo valor de `QUICK` que el notebook 06_deep_encoder: en modo rápido solo se
calculan los subconjuntos pequeños que usa ese notebook, en `outputs/elmo_cache_quick/`.

## Configuración

In [1]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, display

from jailbreak_detection.models.data import SPLITS, load_splits, select_splits, truncate
from jailbreak_detection.models.elmo_cache import (ELMO_DIM, ElmoCache, ElmoTF, cache_status,
                                                   config_name, precompute_split)
from jailbreak_detection.models.training import step1_winner

QUICK = False            # mismo valor que en 06_deep_encoder
FORCE_ABLATIONS = False  # generar también las cachés de las ablaciones aunque ELMo no haya ganado
BATCH_SIZE = 16          # batch de ELMo; si no cabe en la GPU, bajarlo (no cambia los vectores)
SEED = 42

DATA = Path("../data")
TAG = "_quick" if QUICK else ""
ELMO_CACHE = Path(f"../outputs/elmo_cache{TAG}")
DEEP_OUT = Path(f"../outputs/deep_encoder{TAG}")   # resultados de 06_deep_encoder

BASE = ("head_tail", 1024)
ABLATIONS = [("head", 1024), ("head_tail", 512)]
winner = step1_winner(DEEP_OUT)
configs = [BASE] + (ABLATIONS if FORCE_ABLATIONS or winner == "elmo_frozen" else [])
print("Ganador del paso 1 en 06_deep_encoder:", winner or "todavía no hay resultados")
print("Cachés a generar:", [config_name(*c) for c in configs])

Ganador del paso 1 en 06_deep_encoder: elmo_frozen
Cachés a generar: ['head_tail__L1024', 'head__L1024', 'head_tail__L512']


## Tokens a procesar y tamaño en disco

Cada token ocupa 1024 × 2 bytes. La tabla muestra, por configuración y split, los tokens tras el
truncado, el tamaño final y lo que ya está calculado.

In [2]:
splits = select_splits(load_splits(DATA / "interim"), QUICK)

rows = []
for truncation, max_len in configs:
    status = cache_status(ELMO_CACHE, truncation, max_len)
    for split in SPLITS:
        n_tokens = int(sum(len(truncate(t, max_len, truncation)) for t in splits[split]["tokens_flat"]))
        st = status[split]
        rows.append({
            "Configuración": config_name(truncation, max_len), "Split": split,
            "Prompts": len(splits[split]), "Tokens": n_tokens,
            "GB": n_tokens * ELMO_DIM * 2 / 1e9,
            "Hecho": 0 if st is None else st["rows_written"],
            "Completa": bool(st and st["complete"]),
        })
plan = pd.DataFrame(rows)
pending_tokens = int((plan["Tokens"] - plan["Hecho"]).sum())
print(f"Total: {plan['Tokens'].sum():,} tokens ({plan['GB'].sum():.2f} GB); "
      f"pendientes: {pending_tokens:,}")
plan.round(3)

Total: 11,210,162 tokens (22.96 GB); pendientes: 11,210,162


,Configuración,Split,Prompts,Tokens,GB,Hecho,Completa
0,head_tail__L1024,train,9672,2831059,5.798,0,False
1,head_tail__L1024,val,2015,599140,1.227,0,False
2,head_tail__L1024,test,2019,584250,1.197,0,False
3,head__L1024,train,9672,2831059,5.798,0,False
4,head__L1024,val,2015,599140,1.227,0,False
5,head__L1024,test,2019,584250,1.197,0,False
6,head_tail__L512,train,9672,2241657,4.591,0,False
7,head_tail__L512,val,2015,470748,0.964,0,False
8,head_tail__L512,test,2019,468859,0.960,0,False


## Carga de ELMo y estimación del tiempo

Se activa el crecimiento de memoria de la GPU en TensorFlow antes de cargar el modelo. La
estimación se hace midiendo la velocidad sobre un batch de prompts de train de longitud parecida
(es orientativa: la velocidad por token cambia con la longitud; durante el cálculo se muestra el
tiempo restante con la velocidad real).

In [3]:
elmo = ElmoTF(DATA / "models/elmo", batch_size=BATCH_SIZE)
print(f"TensorFlow {elmo.tf_version} | ELMo se ejecuta en: {elmo.device}")
if not elmo.gpus:
    display(HTML(
        '<div style="background:#fde2e1;border:2px solid #c0392b;padding:10px;font-size:15px">'
        '<b>ELMo se va a ejecutar en CPU.</b> Es mucho más lento (del orden de cientos de tokens/s): '
        'mira la estimación de abajo. El cálculo es reanudable, se puede interrumpir y relanzar.</div>'))

# Muestra para medir la velocidad: un batch de prompts de longitud parecida (como se procesan
# realmente, ordenados por longitud), empezando en una posición al azar del orden por longitud
rng = np.random.default_rng(SEED)
train_trunc = [truncate(t, BASE[1], BASE[0]) for t in splits["train"]["tokens_flat"]]
by_length = np.argsort([len(t) for t in train_trunc], kind="stable")
start = int(rng.integers(0, max(1, len(by_length) - BATCH_SIZE)))
sample = [train_trunc[i] for i in by_length[start:start + BATCH_SIZE]]

2026-11-12 21:36:26,071 : INFO : Loading model from ../data/models/elmo...


2026-11-12 21:36:27,568 : INFO : No vocabulary file found in the model.


2026-11-12 21:36:27,571 : INFO : No vocabulary file provided; using special tokens only.


2026-11-12 21:36:27,571 : INFO : We will cache the vocabulary of 3 tokens.


/content/tfenv/lib/python3.13/site-packages/simple_elmo/model.py:531: UserWarning: `tf.nn.rnn_cell.LSTMCell` is deprecated and will be removed in a future version. This class is equivalent as `tf.keras.layers.LSTMCell`, and will be replaced by that in Tensorflow 2.0.
  lstm_cell = tf.compat.v1.nn.rnn_cell.LSTMCell(


TensorFlow 2.21.0 | ELMo se ejecuta en: GPU (1)


In [4]:
speed = elmo.tokens_per_second(sample)
print(f"Velocidad medida: {speed:,.0f} tokens/s sobre {sum(map(len, sample)):,} tokens")
print(f"Estimación para lo pendiente ({pending_tokens:,} tokens): "
      f"{pending_tokens / speed / 3600:.1f} h")

Velocidad medida: 1,559 tokens/s sobre 480 tokens
Estimación para lo pendiente (11,210,162 tokens): 2.0 h


## Cálculo de la caché

Reanudable: si se interrumpe (o se para el kernel), al volver a ejecutar esta celda continúa desde
el último batch guardado. Muestra el avance cada minuto.

In [5]:
for truncation, max_len in configs:
    for split in SPLITS:
        precompute_split(elmo, splits[split], truncation, max_len, ELMO_CACHE, split)

[head_tail__L1024/train] batch 1/605 | 16,384/2,831,059 tokens | 3,020 tokens/s | quedan ~15.5 min


[head_tail__L1024/train] batch 18/605 | 294,912/2,831,059 tokens | 4,504 tokens/s | quedan ~9.4 min


[head_tail__L1024/train] batch 35/605 | 573,440/2,831,059 tokens | 4,486 tokens/s | quedan ~8.4 min


[head_tail__L1024/train] batch 49/605 | 796,550/2,831,059 tokens | 4,159 tokens/s | quedan ~8.2 min


[head_tail__L1024/train] batch 62/605 | 977,504/2,831,059 tokens | 3,844 tokens/s | quedan ~8.0 min


[head_tail__L1024/train] batch 77/605 | 1,159,408/2,831,059 tokens | 3,657 tokens/s | quedan ~7.6 min


... (50 outputs left out)

[head_tail__L512/val] batch 126/126 | 470,748/470,748 tokens | 4,347 tokens/s | quedan ~0.0 min


[head_tail__L512/test] batch 1/127 | 8,192/468,859 tokens | 4,445 tokens/s | quedan ~1.7 min


[head_tail__L512/test] batch 36/127 | 275,650/468,859 tokens | 4,446 tokens/s | quedan ~0.7 min


[head_tail__L512/test] batch 127/127 | 468,859/468,859 tokens | 4,419 tokens/s | quedan ~0.0 min


## Comprobaciones

Sobre una muestra de prompts de validación de la configuración base:

1. Los vectores leídos de la caché (float16 → float32) frente a recalcularlos ahora, cada prompt
   **solo** en su batch: similitud coseno por token. Debe ser ≈ 1 (solo cambia el redondeo a float16).
2. Cada prompt solo frente a todos juntos en un batch: comprueba que el reinicio del estado hace
   que el vector de un prompt no dependa de los demás.

In [6]:
def token_cosines(a, b):
    return (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))


cache = ElmoCache(ELMO_CACHE, *BASE)
val = splits["val"]
check = rng.choice(len(val), size=min(8, len(val)), replace=False)
check_tokens = [truncate(val["tokens_flat"].iloc[i], BASE[1], BASE[0]) for i in check]
together = elmo(check_tokens)

rows = []
for j, i in enumerate(check):
    pid, n = int(val["id"].iloc[i]), len(check_tokens[j])
    alone = elmo([check_tokens[j]])[0, :n]
    cached = cache.get(pid)
    rows.append({"id": pid, "Tokens": n,
                 "Coseno caché vs recalculado (mín)": token_cosines(cached, alone).min(),
                 "Coseno caché vs recalculado (media)": token_cosines(cached, alone).mean(),
                 "Coseno solo vs en batch (mín)": token_cosines(alone, together[j, :n]).min()})
pd.DataFrame(rows).round(6)

,id,Tokens,Coseno caché vs recalculado (mín),Coseno caché vs recalculado (media),Coseno solo vs en batch (mín)
0,3060,575,1.0,1.0,1.0
1,9943,1024,1.0,1.0,1.0
2,6679,46,1.0,1.0,1.0
3,12877,864,1.0,1.0,1.0
4,10612,517,1.0,1.0,1.0
5,1476,594,1.0,1.0,1.0
6,6615,18,1.0,1.0,1.0
7,11729,152,1.0,1.0,1.0


## Tamaño en disco

In [7]:
rows = []
for truncation, max_len in configs:
    for split in SPLITS:
        d = ELMO_CACHE / config_name(truncation, max_len) / split
        size = sum(f.stat().st_size for f in d.glob("*")) if d.exists() else 0
        st = cache_status(ELMO_CACHE, truncation, max_len)[split]
        rows.append({"Configuración": config_name(truncation, max_len), "Split": split,
                     "GB en disco": size / 1e9, "Completa": bool(st and st["complete"]),
                     "Minutos de cálculo": (st or {}).get("seconds", 0) / 60})
print("Espacio libre:", round(shutil.disk_usage(ELMO_CACHE).free / 1e9, 1), "GB")
pd.DataFrame(rows).round(3)

Espacio libre: 83.9 GB


,Configuración,Split,GB en disco,Completa,Minutos de cálculo
0,head_tail__L1024,train,5.798,True,15.133
1,head_tail__L1024,val,1.227,True,2.800
2,head_tail__L1024,test,1.197,True,2.694
3,head__L1024,train,5.798,True,10.743
4,head__L1024,val,1.227,True,2.284
5,head__L1024,test,1.197,True,2.228
6,head_tail__L512,train,4.591,True,8.447
7,head_tail__L512,val,0.964,True,1.805
8,head_tail__L512,test,0.960,True,1.768
